# Task 4 — Deep Learning Fundamentals
**Course:** Machine Learning & Deep Learning  
**Points:** 10/60  
**School of Artificial Intelligence and Data Science**

---

## Overview
- Multi-Layer Perceptron (>=3 hidden layers) with Dropout & Batch Normalization
- Activation experiments: ReLU vs Tanh vs Sigmoid
- Optimizer study: SGD vs Adam vs RMSProp — loss & accuracy curves
- Learning-rate sweep for Adam optimizer
- Conceptual questions (vanishing gradient, BatchNorm, learning rate)
- Dataset A from Task 2 (Gaming_Academic_Performance.csv)


## Step 0 — Imports & Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

tf.random.set_seed(42)
np.random.seed(42)
sns.set_theme(style='whitegrid')

OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = ['Low', 'Medium', 'High']

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f"Outputs save to: {OUTPUT_DIR}")
print(f"Classes: {CLASS_NAMES}")


TensorFlow version: 2.21.0
GPU available: False
Outputs save to: c:\Users\Senya\ml-end\results
Classes: ['Low', 'Medium', 'High']


## Step 1a — Data Loading & Preprocessing

Loads `../data/Gaming_Academic_Performance.csv`.
`__file__` is never used — the notebook portable root is derived from
`os.path.abspath('.')`.


In [2]:
def load_and_preprocess_dataset_A():
    csv_path = os.path.join('..', 'data', 'Gaming_Academic_Performance.csv')
    try:
        df = pd.read_csv(csv_path)
        print(f"Using dataset from {csv_path}")
    except FileNotFoundError:
        print("CSV not found — generating synthetic dataset with same schema...")
        np.random.seed(42)
        n = 1500
        df = pd.DataFrame({
            "student_id": [f"S{i:04d}" for i in range(n)],
            "age": np.random.randint(15, 25, n),
            "gender": np.random.choice(["Male","Female","Other"], n),
            "gaming_hours":    np.random.uniform(0, 12, n),
            "study_hours":     np.random.uniform(0, 10, n),
            "sleep_hours":     np.random.uniform(4, 10, n),
            "attendance":      np.random.uniform(50, 100, n),
            "gaming_genre":    np.random.choice(["Action","RPG","Sports","Strategy","Puzzle"], n),
            "social_activity": np.random.uniform(1, 10, n),
            "device_usage":    np.random.uniform(1, 12, n),
            "reaction_time_ms":np.random.uniform(150, 500, n),
            "addiction_score": np.random.uniform(1, 10, n),
            "stress_level":    np.random.choice(["Low","Medium","High"], n),
        })
        df["grades"] = (40 + df["study_hours"]*4 - df["gaming_hours"]*1.5 +
                         df["sleep_hours"]*1.5 + df["attendance"]*0.2 +
                         np.random.normal(0, 8, n)).clip(0, 100)
    print(f"Shape: {df.shape}")
    print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum()>0]}")
    df.drop(columns=["student_id"], errors="ignore", inplace=True)
    for col in df.select_dtypes(include="number").columns:
        df[col].fillna(df[col].median(), inplace=True)
    for col in df.select_dtypes(include="object").columns:
        df[col].fillna(df[col].mode()[0], inplace=True)
    le = LabelEncoder()
    for col in df.select_dtypes(include="object").columns:
        df[col] = le.fit_transform(df[col])
    df["performance"] = pd.cut(df["grades"], bins=[0,50,75,100], labels=["Low","Medium","High"])
    df["performance"] = LabelEncoder().fit_transform(df["performance"].astype(str))
    df.drop(columns=["grades"], inplace=True)
    print(f"\nClass distribution:\n{pd.Series(df['performance']).value_counts()}")
    return df.drop(columns=["performance"]), df["performance"]


## Step 1b — Train / Val / Test Split + One-Hot Encode

In [3]:
def prepare_data():
    X_raw, y_raw = load_and_preprocess_dataset_A()
    X = X_raw.astype(np.float32)
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    NUM_CLASSES = len(np.unique(y_raw))
    y_ohe = tf.keras.utils.to_categorical(y_raw, num_classes=NUM_CLASSES)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y_ohe, test_size=0.30, random_state=42, stratify=y_raw)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42)
    print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
    print(f"Features: {X_train.shape[1]}, Classes: {NUM_CLASSES}")
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = prepare_data()
INPUT_DIM = X_train.shape[1]
N_CLASSES = y_train.shape[1]


Using dataset from ..\data\Gaming_Academic_Performance.csv
Shape: (8000, 14)

Missing values:
Series([], dtype: int64)

Class distribution:
performance
0    2931
2    2924
1    2010
3     135
Name: count, dtype: int64
Train: (5600, 12)  Val: (1200, 12)  Test: (1200, 12)
Features: 12, Classes: 4


## Step 2 — MLP Architecture (>= 3 Hidden Layers)

```
Input(12) -> Dense(128) -> BN -> ReLU  -> Dropout(0.4)
          -> Dense(64)  -> BN -> ReLU  -> Dropout(0.3)
          -> Dense(32)  -> BN -> ReLU  -> Dropout(0.2)  <-- 3rd hidden layer
          -> Dense(4)   -> Softmax
```

Dropout + BatchNormalisation at every hidden layer.
L2 regularisation (lambda=1e-4) on all hidden Dense layers.


In [4]:
def build_mlp(input_dim, n_classes, activation="relu"):
    return keras.Sequential([
        layers.Input(shape=(input_dim,)),
        # Hidden layer 1
        layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Activation(activation),
        layers.Dropout(0.4),
        # Hidden layer 2
        layers.Dense(64, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Activation(activation),
        layers.Dropout(0.3),
        # Hidden layer 3  (satisfies >= 3 hidden layer requirement)
        layers.Dense(32, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Activation(activation),
        layers.Dropout(0.2),
        # Output layer
        layers.Dense(n_classes, activation="softmax"),
    ], name="mlp_classifier")

model = build_mlp(INPUT_DIM, N_CLASSES, activation="relu")
model.summary()


Model: "mlp_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,028 (50.89 KB)

 Trainable params: 12,580 (49.14 KB)

 Non-trainable params: 448 (1.75 KB)

## Step 2b — Activation Function Comparison

Demonstrates: ReLU, Tanh, Sigmoid — at least two different activations (ReLU + Tanh),
plus Sigmoid as a third.


In [5]:
def train_mlp_activation(activation, X_train, X_val, X_test,
                         y_train, y_val, y_test,
                         epochs=2, cbs=None):
    m = build_mlp(INPUT_DIM, N_CLASSES, activation=activation)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
    hist = m.fit(X_train, y_train, epochs=epochs, batch_size=32,
                 validation_data=(X_val, y_val),
                 callbacks=cbs or [], verbose=0)
    _, test_acc = m.evaluate(X_test, y_test, verbose=0)
    print(f"  {activation:<8s}:  Accuracy={test_acc:.4f}  "
          f"Final val_acc={hist.history['val_accuracy'][-1]:.4f}")
    return m, hist

early_stop = callbacks.EarlyStopping(
    monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=0)

act_results = {}
for act in ["relu", "tanh", "sigmoid"]:
    m_a, h_a = train_mlp_activation(act, X_train, X_val, X_test,
                                    y_train, y_val, y_test,
                                    cbs=[early_stop])
    act_results[act] = {"model": m_a, "history": h_a}

# Validation accuracy comparison plot
fig, ax = plt.subplots(figsize=(7, 4))
for act, data in act_results.items():
    ep = range(1, len(data["history"].history["val_accuracy"]) + 1)
    ax.plot(ep, data["history"].history["val_accuracy"],
            label=act.title(), lw=2)
ax.set_title("Validation Accuracy by Activation Function (3-layer MLP)")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Accuracy")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "T4_activation_comparison.png"), dpi=150)
plt.close()

df_act = pd.DataFrame(
    {act: [act_results[act]["model"].evaluate(X_test, y_test, verbose=0)[1]]
     for act in act_results}, index=["Test Accuracy"]).T
print("\nActivation comparison — Test Accuracy:")
print(df_act.round(4).to_string())


  relu    :  Accuracy=0.8175  Final val_acc=0.8175
  tanh    :  Accuracy=0.7600  Final val_acc=0.8017
  sigmoid :  Accuracy=0.8217  Final val_acc=0.8250

Activation comparison — Test Accuracy:
         Test Accuracy
relu            0.8175
tanh            0.7600
sigmoid         0.8217


## Step 3 — Optimizer Comparison (SGD vs Adam vs RMSProp)

Same MLP architecture; only the optimizer varies.
SGD (lr=0.01, momentum=0.9); Adam (lr=0.001); RMSprop (lr=0.001).


In [6]:
def train_with_optimizer(optimizer_name, lr):
    m = build_mlp(INPUT_DIM, N_CLASSES, activation="relu")
    if optimizer_name == "SGD":
        opt = keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    elif optimizer_name == "Adam":
        opt = keras.optimizers.Adam(learning_rate=lr)
    else:
        opt = keras.optimizers.RMSprop(learning_rate=lr)
    m.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])
    hist = m.fit(
        X_train, y_train, epochs=2, batch_size=32,
        validation_data=(X_val, y_val),
        callbacks=[callbacks.EarlyStopping(
            patience=10, restore_best_weights=True, verbose=0)],
        verbose=0)
    _, test_acc = m.evaluate(X_test, y_test, verbose=0)
    print(f"{optimizer_name:<8s}: Test Accuracy = {test_acc:.4f}")
    return m, hist

results = {}
for name, lr in [("SGD", 0.01), ("Adam", 0.001), ("RMSprop", 0.001)]:
    _, hist = train_with_optimizer(name, lr)
    results[name] = {"history": hist}


SGD     : Test Accuracy = 0.8317
Adam    : Test Accuracy = 0.8083
RMSprop : Test Accuracy = 0.8092


In [7]:
# Convergence speed — epochs to plateau
def plateau_epoch(h):
    v = h["val_accuracy"]
    best, patience = v[0], 0
    for i, val in enumerate(v):
        if val > best:
            best, patience = val, 0
        else:
            patience += 1
        if patience >= 5:
            return i + 1
    return len(v)

print("\nConvergence speed (epochs to plateau):")
for name, data in results.items():
    ep = plateau_epoch(data["history"].history)
    print(f"  {name:<8s}: plateau epoch={ep}")



Convergence speed (epochs to plateau):
  SGD     : plateau epoch=2
  Adam    : plateau epoch=2
  RMSprop : plateau epoch=2


In [8]:
# ┌─ Accuracy curves ──────────────────────────────────────────────────────┐
fig_a, ax_a = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, data) in zip(ax_a, results.items()):
    h = data["history"].history
    ax.plot(h["accuracy"],     "b-",  label="Train Acc", lw=2)
    ax.plot(h["val_accuracy"], "b--", label="Val Acc",   lw=2)
    ax.set_title(f"{name}  -- Accuracy", fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle("Task 4  -- Optimizer Accuracy Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "T4_optimizer_accuracy.png"), dpi=150)
plt.close()

# ┌─ Loss curves ──────────────────────────────────────────────────────────┐
fig_l, ax_l = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, data) in zip(ax_l, results.items()):
    h = data["history"].history
    ax.plot(h["loss"],     "r-",  label="Train Loss", lw=2)
    ax.plot(h["val_loss"], "r--", label="Val Loss",   lw=2)
    ax.set_title(f"{name}  -- Loss", fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle("Task 4  -- Optimizer Loss Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "T4_optimizer_loss.png"), dpi=150)
plt.close()

# ┌─ Combined 2x3 layout ─────────────────────────────────────────────────┐
fig_c, ax_c = plt.subplots(2, 3, figsize=(18, 10))
for col, (name, data) in enumerate(results.items()):
    h = data["history"].history
    ax_c[0, col].plot(h["accuracy"],     "b-",  label="Train Acc", lw=2)
    ax_c[0, col].plot(h["val_accuracy"], "b--", label="Val Acc",   lw=2)
    ax_c[0, col].set_title(f"{name}  -- Accuracy", fontweight="bold")
    ax_c[0, col].set_xlabel("Epoch"); ax_c[0, col].set_ylabel("Accuracy")
    ax_c[0, col].legend(); ax_c[0, col].grid(alpha=0.3)
    ax_c[1, col].plot(h["loss"],     "r-",  label="Train Loss", lw=2)
    ax_c[1, col].plot(h["val_loss"], "r--", label="Val Loss",   lw=2)
    ax_c[1, col].set_title(f"{name}  -- Loss", fontweight="bold")
    ax_c[1, col].set_xlabel("Epoch"); ax_c[1, col].set_ylabel("Loss")
    ax_c[1, col].legend(); ax_c[1, col].grid(alpha=0.3)
plt.suptitle("Task 4  -- Optimizer Comparison: Loss and Accuracy Curves",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "T4_optimizer_full_history.png"), dpi=150)
plt.close()

print("\nSaved: T4_optimizer_accuracy.png, T4_optimizer_loss.png, T4_optimizer_full_history.png")



Saved: T4_optimizer_accuracy.png, T4_optimizer_loss.png, T4_optimizer_full_history.png


## Step 4 — Learning Rate Study (Adam, 4 LR values)

In [9]:
lrs = [0.1, 0.01, 0.001, 0.0001]
adam_accs = []
for lr in lrs:
    m_lr, _ = train_mlp_activation("relu", X_train, X_val, X_test,
                                    y_train, y_val, y_test,
                                    cbs=[callbacks.EarlyStopping(
                                        patience=6, restore_best_weights=True, verbose=0)])
    _, acc = m_lr.evaluate(X_test, y_test, verbose=0)
    adam_accs.append(acc)
    print(f"LR={lr:.4f}: Accuracy={acc:.4f}")

plt.figure(figsize=(8, 4))
plt.semilogx(lrs, adam_accs, "o-", lw=2, color="navy")
plt.xlabel("Learning Rate (log scale)")
plt.ylabel("Test Accuracy")
plt.title("Learning Rate Effect on Adam Optimizer")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "T4_learning_rate_effect.png"), dpi=150)
plt.close()
print("Saved T4_learning_rate_effect.png")


  relu    :  Accuracy=0.8183  Final val_acc=0.8167
LR=0.1000: Accuracy=0.8183
  relu    :  Accuracy=0.8017  Final val_acc=0.8100
LR=0.0100: Accuracy=0.8017
  relu    :  Accuracy=0.8183  Final val_acc=0.8142
LR=0.0010: Accuracy=0.8183
  relu    :  Accuracy=0.8208  Final val_acc=0.8308
LR=0.0001: Accuracy=0.8208
Saved T4_learning_rate_effect.png


## Step 5 — Final Model Evaluation (Adam, LR=0.001)

In [10]:
m_final, _ = train_mlp_activation("relu", X_train, X_val, X_test,
                                   y_train, y_val, y_test,
                                   cbs=[callbacks.EarlyStopping(
                                       patience=10, restore_best_weights=True, verbose=0)])

y_pred_raw  = m_final.predict(X_test, verbose=0)
y_pred_cls  = np.argmax(y_pred_raw, axis=1)
y_true_cls  = np.argmax(y_test,     axis=1)

print(classification_report(y_true_cls, y_pred_cls,
                            labels=[0, 1, 2], target_names=CLASS_NAMES))

cm = confusion_matrix(y_true_cls, y_pred_cls, labels=[0, 1, 2])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix — MLP (Adam lr=0.001)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "T4_confusion_matrix.png"), dpi=150)
plt.close()
print("Saved T4_confusion_matrix.png")


  relu    :  Accuracy=0.8017  Final val_acc=0.8050
              precision    recall  f1-score   support

         Low       0.79      0.93      0.85       418
      Medium       0.78      0.92      0.84       305
        High       0.85      0.64      0.73       453

   micro avg       0.80      0.82      0.81      1176
   macro avg       0.80      0.83      0.81      1176
weighted avg       0.81      0.82      0.80      1176

Saved T4_confusion_matrix.png


## Step 6 — Conceptual Questions

**Q1 — Vanishing Gradient Problem**
In deep networks ∂L/∂w at early layers is the product of many chain-rule terms.
With sigmoid (|σ'|<=0.25) or tanh (|σ'|<=1) this shrinks exponentially with depth,
making early layers learn very slowly or not at all.
Solutions: ReLU (gradient=1 for x>0), BatchNorm, residual connections in ResNet,
LayerNorm/GroupNorm in transformers.

**Q2 — Batch Normalisation formula** (per feature channel, mini-batch B):
```
  mu_B   = (1/m) Σ_i x^(i)             [batch mean]
  sigma^2= (1/m) Σ_i (x^(i)-mu_B)^2    [batch variance]
  xhat   = (x^(i)-mu_B)/sqrt(sigma^2+epsilon)
  y      = gamma*xhat + beta            [scale + shift, learnable]
```
BN reduces internal covariate shift, permits higher learning rates,
faster convergence, and acts as a mild regulariser.

**Q3 — Learning Rate**
| Setting | Effect |
|---|---|
|Too high (>0.1)       |Overshoot minima, oscillate, diverge|
|Too low (<1e-5)       |Slow convergence, local-minimum traps|
|Just right            |Stable, fast convergence|
Good schedules: ReduceLROnPlateau, CosineDecay, Warm-up.
